# CineScope — Baseline Validation

Reads the bronze `movies_ratings` Parquet table from the data root. Does **not** re-run raw ETL.

Charts are shown inline and also written under `outputs/charts/generated/`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from dotenv import load_dotenv

REPO = Path.cwd()
if not (REPO / "src").exists():
    REPO = Path.cwd().parent
sys.path.insert(0, str(REPO / "src"))
load_dotenv(REPO / ".env")

from pyspark.sql import functions as F

from cinescope.paths import get_paths
from cinescope.spark_session import build_spark_session

paths = get_paths(create_dirs=False, validate_mount=True)
spark = build_spark_session(app_name="cinescope-baseline-notebook", paths=paths)
charts_dir = REPO / "outputs" / "charts" / "generated"
charts_dir.mkdir(parents=True, exist_ok=True)

def save_and_show(fig, path: Path):
    fig.savefig(path, dpi=120, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("wrote", path)

print("data root:", paths.data_root)
print("Parquet:", paths.movies_ratings_dir)

In [ ]:
df = spark.read.parquet(str(paths.movies_ratings_dir))
df.printSchema()
row_count = df.count()
print("row_count:", row_count)
df.show(10, truncate=False)

In [ ]:
df.select(
    F.count("*").alias("n"),
    F.avg("average_rating").alias("avg_rating"),
    F.avg("num_votes").alias("avg_votes"),
    F.avg("runtime_minutes").alias("avg_runtime"),
    F.min("start_year").alias("min_year"),
    F.max("start_year").alias("max_year"),
).show()

null_exprs = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]
df.agg(*null_exprs).show()

## Visuals

In [ ]:
top_genres = (
    df.select(F.explode_outer("genres").alias("genre"))
    .filter(F.col("genre").isNotNull())
    .groupBy("genre")
    .count()
    .orderBy(F.desc("count"))
    .limit(15)
    .toPandas()
)
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(top_genres["genre"][::-1], top_genres["count"][::-1])
ax.set_xlabel("Films")
ax.set_title("Top genres in bronze movies_ratings")
fig.tight_layout()
save_and_show(fig, charts_dir / "baseline_top_genres.png")
top_genres

In [ ]:
by_decade = (
    df.filter(F.col("start_year").isNotNull())
    .withColumn("decade", (F.floor(F.col("start_year") / 10) * 10).cast("int"))
    .groupBy("decade")
    .agg(
        F.count("*").alias("movies"),
        F.avg("average_rating").alias("avg_rating"),
        F.avg("num_votes").alias("avg_votes"),
    )
    .orderBy("decade")
    .toPandas()
)
fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.bar(by_decade["decade"], by_decade["movies"], width=8, alpha=0.4, label="movies")
ax2 = ax1.twinx()
ax2.plot(by_decade["decade"], by_decade["avg_rating"], color="C3", marker="o", label="avg rating")
ax1.set_xlabel("Decade")
ax1.set_ylabel("Movies")
ax2.set_ylabel("Avg rating")
ax1.set_title("Films and average rating by decade")
fig.tight_layout()
save_and_show(fig, charts_dir / "baseline_by_decade.png")
by_decade.tail(10)

In [ ]:
rating_pd = (
    df.select("average_rating", "num_votes", "runtime_minutes")
    .sample(False, 0.05, seed=1)
    .toPandas()
)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(rating_pd["average_rating"].dropna(), bins=30, color="C0", alpha=0.85)
axes[0].set_title("Rating distribution (5% sample)")
axes[0].set_xlabel("average_rating")
axes[1].scatter(
    rating_pd["runtime_minutes"].clip(upper=300),
    rating_pd["average_rating"],
    s=6,
    alpha=0.2,
)
axes[1].set_title("Runtime vs rating (5% sample)")
axes[1].set_xlabel("runtime_minutes (clipped at 300)")
axes[1].set_ylabel("average_rating")
fig.tight_layout()
save_and_show(fig, charts_dir / "baseline_rating_runtime.png")
spark.stop()